# Lab 21 solution, notebook 2: a second name on the gateway, and both deployments broken

The worked answer to the challenge of `21b_litellm_at_the_desk.ipynb`. The setup repeats the notebook's lookups,
its agent and its two gateway functions, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import json
import os
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id."""
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

In [3]:
find_customer_tool = {
    "type": "function",
    "function": {
        "name": "find_customer",
        "description": "Find a shop customer by full name. Returns a list of matches with their customer_id.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "First and last name, for example 'Ana Silva'"}},
            "required": ["name"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

get_invoices_tool = {
    "type": "function",
    "function": {
        "name": "get_invoices",
        "description": "Count a customer's invoices and add up their total in US dollars.",
        "parameters": {
            "type": "object",
            "properties": {"customer_id": {"type": "integer"}},
            "required": ["customer_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

get_support_rep_tool = {
    "type": "function",
    "function": {
        "name": "get_support_rep",
        "description": "Name the support rep who looks after a customer.",
        "parameters": {
            "type": "object",
            "properties": {"customer_id": {"type": "integer"}},
            "required": ["customer_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

TOOLS = [find_customer_tool, get_invoices_tool, get_support_rep_tool]
FUNCTIONS = {"find_customer": find_customer, "get_invoices": get_invoices, "get_support_rep": get_support_rep}

In [4]:
from openai import OpenAI, omit

INSTRUCTIONS = (
    "You help the support reps of a music shop. Look up every fact with the tools; "
    "never answer from memory. Answer in one or two sentences."
)


def ask_desk(client, model, question):
    """Answer a rep's question: call the model, run the lookups it asks for, repeat until it answers."""
    messages = [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": question}]

    while True:
        reply = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            reasoning_effort=EFFORT or omit,
        )
        message = reply.choices[0].message
        if not message.tool_calls:
            return message.content

        messages.append(message)
        for call in message.tool_calls:
            arguments = json.loads(call.function.arguments)
            result = FUNCTIONS[call.function.name](**arguments)
            print("  lookup:", call.function.name, arguments)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

In [5]:
import socket
import subprocess
import sys
import time

from openai import APIConnectionError

# The first free port, starting at LiteLLM's usual 4000
PORT = 4000
while True:
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", PORT)) != 0:
            break
    print("port", PORT, "is in use, trying the next one")
    PORT += 1

GATEWAY = f"http://127.0.0.1:{PORT}"
GATEWAY_KEY = "lab-21-local-key"
print("the gateway will use port", PORT)


def start_gateway():
    """Start the LiteLLM gateway with our config file, and return its process once it answers."""
    # 1. Is our port still free? It is not when a gateway we started is still running
    with socket.socket() as probe:
        port_in_use = probe.connect_ex(("127.0.0.1", PORT)) == 0
    assert not port_in_use, f"Port {PORT} is in use. Stop the running gateway first: stop_gateway(gateway)."

    # 2. Start the gateway as a second program; what it prints goes to outputs/gateway.log
    with open(OUTPUTS / "gateway.log", "w") as log:
        process = subprocess.Popen(
            [sys.executable, "-m", "litellm.proxy.proxy_cli", "--config", str(CONFIG_FILE), "--host", "127.0.0.1", "--port", str(PORT)],
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    # 3. Ask for its model list once a second until it answers
    checker = OpenAI(base_url=GATEWAY + "/v1", api_key=GATEWAY_KEY, max_retries=0)
    while True:
        assert process.poll() is None, "The gateway stopped while starting. Read outputs/gateway.log."
        try:
            checker.models.list()
            return process
        except APIConnectionError:
            time.sleep(1)


def stop_gateway(process):
    """Ask the gateway to stop, and wait until it has."""
    process.terminate()
    process.wait()

the gateway will use port 4000


In [6]:
from openai import DefaultHttpxClient

gateway_headers = []


def keep_headers(response):
    """Called by the client for every HTTP reply: keep the reply's headers."""
    gateway_headers.append(response.headers)


logged_client = OpenAI(
    base_url=GATEWAY + "/v1",
    api_key=GATEWAY_KEY,
    timeout=60,
    http_client=DefaultHttpxClient(event_hooks={"response": [keep_headers]}),
)

## Task 1: a second job with its own name

The config of Part B with a third deployment: the name `notes`, the id `notes-primary`. It points at the same
model as `desk`; what is new is the name, which a program can ask for and the gateway can count separately.
`OUTPUTS` is a folder beside this file, so this config does not replace the notebook's.

In [7]:
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
CONFIG_FILE = OUTPUTS / "gateway_config.yaml"

OPENAI_ADDRESS = "https://api.openai.com/v1"

config = f"""model_list:
  - model_name: desk
    litellm_params:
      model: openai/{MODEL}
      api_key: os.environ/OPENAI_API_KEY
      api_base: {OPENAI_ADDRESS}
    model_info:
      id: desk-primary
  - model_name: desk-backup
    litellm_params:
      model: openai/{settings["MODEL_STRONG"]}
      api_key: os.environ/OPENAI_API_KEY
      api_base: {OPENAI_ADDRESS}
    model_info:
      id: desk-backup-1
  - model_name: notes
    litellm_params:
      model: openai/{MODEL}
      api_key: os.environ/OPENAI_API_KEY
    model_info:
      id: notes-primary

router_settings:
  fallbacks:
    - desk: [desk-backup]

general_settings:
  master_key: lab-21-local-key
"""

CONFIG_FILE.write_text(config)

gateway = start_gateway()
gateway_client = OpenAI(base_url=GATEWAY + "/v1", api_key=GATEWAY_KEY, timeout=60)
print("the gateway offers:", [model.id for model in gateway_client.models.list()])

the gateway offers: ['desk', 'desk-backup', 'notes']


We ask `notes` for the summary through `with_raw_response`, which gives us the reply's headers. `raw.parse()`
turns the raw reply into the usual object with `choices`.

In [8]:
HELENA = "How many invoices does Helena Holý have, what do they add up to, and who is her support rep?"

raw = gateway_client.chat.completions.with_raw_response.create(
    model="notes",
    messages=[{"role": "user", "content": "Summarise this support question in five words for a ticket log: " + HELENA}],
    reasoning_effort=EFFORT or omit,
)

print(raw.parse().choices[0].message.content)
for header in ["x-litellm-model-group", "x-litellm-model-id", "x-litellm-attempted-fallbacks", "x-litellm-response-cost"]:
    print(header + ":", raw.headers.get(header))
print("the cost in dollars:", f"${float(raw.headers['x-litellm-response-cost']):.8f}")

Helena Holý invoice count, total, representative
x-litellm-model-group: notes
x-litellm-model-id: notes-primary
x-litellm-attempted-fallbacks: 0
x-litellm-response-cost: 1.0799999999999998e-05
the cost in dollars: $0.00001080


The gateway offers three names. The summary came back as *Helena Holý invoice count, total, representative*, and the headers show the name `notes`, the deployment `notes-primary`, 0 fallbacks and a cost of \$0.00001080.

The desk's two jobs now have two names. The gateway reports each request under the name that was asked for, so
the cost of the summaries can be told apart from the cost of the answers, and `notes` can later move to a cheaper
model by changing one line of the config.

## Task 2: break the backup too

This config gave both `desk` and `desk-backup` an `api_base` line, so one `replace` points both at the dead
address. `notes` has no such line and keeps working. We restart the gateway and ask Helena's question inside `try`
and `except`, with a timer.

In [9]:
NOWHERE = "http://127.0.0.1:9/v1"

stop_gateway(gateway)
CONFIG_FILE.write_text(config.replace(OPENAI_ADDRESS, NOWHERE))
gateway = start_gateway()

gateway_headers.clear()
start = time.time()

try:
    print(ask_desk(logged_client, "desk", HELENA))
except Exception as error:
    print(type(error).__name__, "|", error)

print("seconds:", round(time.time() - start, 1))
print("x-litellm-model-id in the reply's headers:", gateway_headers[-1].get("x-litellm-model-id"))

InternalServerError | Error code: 500 - litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.

LiteLLM: model group 'desk' failed with the error above. Fallback to desk-backup also failed: litellm.InternalServerError: InternalServerError: OpenAIException - Connection error. LiteLLM Retried: 2 times, LiteLLM Max Retries: 2
seconds: 29.2
x-litellm-model-id in the reply's headers: desk-backup-1


No `lookup:` line and no answer. The error is an `InternalServerError` with the code 500, sent by the gateway, and its text says what happened: the model group `desk` failed with a connection error, and *Fallback to desk-backup also failed*. The header names `desk-backup-1`, the id of the last deployment the gateway tried.

It took 29.2 seconds. The gateway tried `desk` and then `desk-backup`, each with its two retries, before it gave up. Then the OpenAI client, which by default sends a request again twice after a 500 error, repeated the request, and the gateway went through the same steps each time: three rounds of about ten seconds.

**What should the rep see?** Not this error text. The place for it is the model call inside `ask_desk`: wrap
`client.chat.completions.create(...)` in `try` and `except`, and when it fails return a plain sentence such as
*The assistant cannot reach its model right now. Please look the customer up in the shop system.* The lookups are
our own code and still work, so a desk could also fall back to showing the three lookups' raw results for a
customer the rep names. When the last entry in the fallback list fails too, the gateway returns an error, so the program
needs an answer of its own.

We stop the gateway, which frees its port.

In [10]:
stop_gateway(gateway)
print("gateway stopped")

gateway stopped
